### Validate Files

In [ ]:
import sys
from debate_eval.cli import main

_old_argv = sys.argv[:]
try:
    sys.argv = ["debate_eval.cli", "--validate-only"]
    exit_code = main()
finally:
    sys.argv = _old_argv

print(f"Exit code: {exit_code}")

### One on One Testing

In [ ]:
import sys
from debate_eval.cli import main

affirmative = "students/agent_q.py"
negative = "students/agent_yx.py"
material = "material_001_congestion_pricing.txt"

_old_argv = sys.argv[:]
try:
    sys.argv = [
        "debate_eval.cli",
        "--affirmative-file",
        affirmative,
        "--negative-file",
        negative,
        "--material",
        material,
        "--rounds",
        "5",
        "--turn-time-limit",
        "0",
    ]
    stat_summary = main()
finally:
    sys.argv = _old_argv


### Batch Testing

In [ ]:
import json
from concurrent.futures import ThreadPoolExecutor

from debate_eval.cli import build_parser, run_cli

agents = [
    ("students/agent_yx.py", "students/agent_q.py"),
    ("students/agent_q.py", "students/agent_yx.py"),
    ("students/agent_yx.py", "students/debater.py"),
    ("students/debater.py", "students/agent_yx.py"),
    ("students/agent_q.py", "students/debater.py"),
    ("students/debater.py", "students/agent_q.py"),
]
material = "material_001_congestion_pricing.txt"


def run_match(pair):
    affirmative, negative = pair
    parser = build_parser()
    argv = [
        "debate_eval.cli",
        "--affirmative-file",
        affirmative,
        "--negative-file",
        negative,
        "--material",
        material,
        "--rounds",
        "5",
        "--turn-time-limit",
        "0",
    ]
    return run_cli(parser.parse_args(argv[1:]))


with ThreadPoolExecutor(max_workers=5) as executor:
    stat_summaries = list(executor.map(run_match, agents))

for stat_summary in stat_summaries:
    shorten_stat_summary = {
        "affirmative_name": stat_summary["affirmative_name"],
        "negative_name": stat_summary["negative_name"],
        "winner": stat_summary["winner"],
        "judge_votes": stat_summary["judge_votes"],
        "usage": stat_summary["usage"],
    }
    print(json.dumps(shorten_stat_summary, indent=2))